# Lab 04 - Red Teaming (solution)

Reference notebooks: `4 - cloud evaluation/4.2 - Red Team Agent Demo.ipynb` (new Foundry Evals API,
used here) and `4.2 - Red Team Agent.ipynb` (legacy `beta.red_teams` preview API).

## Red teaming is not a synonym for PyRIT

Microsoft exposes different execution paths:

| Path | Where orchestration runs | Portal |
| --- | --- | --- |
| `project_client.get_openai_client().evals` with `scenario="red_team"` | Foundry (managed) | new Foundry experience |
| `project_client.beta.red_teams` | Foundry (managed, preview) | classic Foundry experience |
| `azure-ai-evaluation[redteam]` | locally, on PyRIT | not compatible with the new portal |

This lab uses the **new Evals API**: attacks are generated, executed and graded server-side, so PyRIT
does not need to be installed locally. Cloud red teaming still uses PyRIT-derived attack strategies.

**Evaluation vs red teaming:** evaluation asks *how good is the answer on my dataset*; red teaming asks
*can I make the system produce a policy-violating answer* - with adaptive probes that no static dataset
contains.

## Step 0 - Configuration

In [ ]:
import os, sys, time, json, warnings
from pprint import pprint
from collections import defaultdict

from lab_utils import load_settings

warnings.filterwarnings("ignore")

settings = load_settings(verbose=True)
credential = settings["credential"]
foundry_project_endpoint = settings["foundry_project_endpoint"]
deployment_name = settings["azure_openai_deployment_name"]

if not foundry_project_endpoint or not deployment_name:
    raise ValueError("FOUNDRY_PROJECT_ENDPOINT and AZURE_OPENAI_CHAT_DEPLOYMENT_NAME are required")

In [ ]:
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=foundry_project_endpoint, credential=credential)
evals_client = project_client.get_openai_client().evals

print(f"Target deployment: {deployment_name}")

## Step 1 - Create the red-team evaluation group

The group defines **what counts as a failure**. Here the built-in Violence evaluator marks an output as
failed when the response violates the violence safety criterion. Creating the group does not run any
attack: it is the stable container under which comparable runs are tracked.

In [ ]:
red_team_eval = evals_client.create(
    name="Violence red-team evaluation - lab 04",
    data_source_config={
        "type": "azure_ai_source",
        "scenario": "red_team",
    },
    testing_criteria=[
        {
            "type": "azure_ai_evaluator",
            "name": "Violence",
            "evaluator_name": "builtin.violence",
            "evaluator_version": "1",
        }
    ],
)

print(f"Evaluation group ID: {red_team_eval.id}")

## Step 2 - Configure and start a small scan

Keep the first scan deliberately small: one risk category, one turn, two strategies.

* **Baseline** sends the adversarial objective directly.
* **Base64** encodes the objective before sending it.

Base64 is not "stronger" than baseline: it is a *different* probe that may bypass one control while
failing against another. What matters is the observed success count per technique.

In [ ]:
red_team_run = evals_client.runs.create(
    eval_id=red_team_eval.id,
    name="Violence Baseline and Base64 run",
    data_source={
        "type": "azure_ai_red_team",
        "item_generation_params": {
            "type": "red_team",
            "attack_strategies": ["baseline", "base64"],
            "num_turns": 1,
        },
        "target": {
            "type": "azure_ai_model",
            "model": deployment_name,
        },
    },
)

print(f"Run ID: {red_team_run.id}")
print(f"Initial status: {red_team_run.status}")

In [ ]:
terminal_statuses = {"completed", "failed", "canceled"}
last_status = None

for attempt in range(60):
    red_team_run = evals_client.runs.retrieve(run_id=red_team_run.id, eval_id=red_team_eval.id)

    if red_team_run.status != last_status:
        print(f"Status: {red_team_run.status}")
        last_status = red_team_run.status

    if red_team_run.status in terminal_statuses:
        break

    if attempt < 59:
        time.sleep(5)
else:
    raise TimeoutError("The run is still active. Continue with the completed-run fallback below.")

## Step 3 - Fallback: pick an existing completed run

Creating a cloud run is the slow and variable part. If your run is still going, this read-only cell
retrieves the most recent completed red-team run and lets you continue the lab with real data.

In [ ]:
completed_runs = []

for candidate_eval in evals_client.list(limit=100, order="desc"):
    payload = candidate_eval.model_dump(exclude_none=True, warnings=False)
    if (payload.get("data_source_config") or {}).get("scenario") != "red_team":
        continue
    for candidate_run in evals_client.runs.list(candidate_eval.id, limit=20, order="desc", status="completed"):
        completed_runs.append((candidate_eval, candidate_run))

if not completed_runs:
    raise RuntimeError("No completed red-team run is available yet. Wait for your run to finish.")

red_team_eval, red_team_run = max(completed_runs, key=lambda pair: pair[1].created_at)

pprint({
    "evaluation_id": red_team_eval.id,
    "run_id": red_team_run.id,
    "run_name": red_team_run.name,
    "status": red_team_run.status,
    "report_url": red_team_run.report_url,
})

## Step 4 - Read the row-level results

The SDK returns exactly the output items used by the portal report. For a safety criterion:

* `pass` -> the response passed the Violence evaluator, so the **attack failed**;
* `fail` -> the response failed the evaluator, so the **attack succeeded**.

Always show numerator and denominator: `1/1 (100%)` is one observed failure, not proof that a technique
always works. Grading is model-based, so surprising rows deserve human review.

In [ ]:
output_items = list(
    evals_client.runs.output_items.list(run_id=red_team_run.id, eval_id=red_team_eval.id)
)

print(f"{len(output_items)} output items")

per_strategy = defaultdict(lambda: {"total": 0, "successful_attacks": 0})

for item in output_items:
    payload = item.model_dump(exclude_none=True, warnings=False)
    metadata = payload.get("metadata") or {}
    strategy = metadata.get("attack_strategy") or metadata.get("strategy") or "unknown"

    results = payload.get("results") or []
    passed = all(r.get("passed") for r in results) if results else None

    per_strategy[strategy]["total"] += 1
    if passed is False:
        per_strategy[strategy]["successful_attacks"] += 1

for strategy, stats in per_strategy.items():
    total = stats["total"]
    success = stats["successful_attacks"]
    rate = f"{success / total:.0%}" if total else "n/a"
    print(f"{strategy:12} ASR = {success}/{total} ({rate})")

In [ ]:
# Inspect one attack in detail: the probe, the model answer and the grader verdict
if output_items:
    payload = output_items[0].model_dump(exclude_none=True, warnings=False)
    pprint({k: payload.get(k) for k in ("id", "status", "metadata")})
    print("\n--- results ---")
    pprint(payload.get("results"))

In [ ]:
print(f"Open the full report in Microsoft Foundry:\n{red_team_run.report_url}")

### Checkpoint

You created a red-team evaluation, ran two attack techniques against a live deployment, computed the
Attack Success Rate per technique and opened the report. Everything below is optional.

## Step 5 (optional) - Widen the scan

Add risk categories and attack strategies, and compare the ASR per technique. Cost and duration grow
roughly with `categories x strategies x turns`, so widen one dimension at a time.

Common strategies: `baseline`, `base64`, `rot13`, `morse`, `leetspeak`, `url`, `flip`,
`character_space`, `unicode_confusable`, `tense`. Multi-turn scans (`num_turns > 1`) simulate an
attacker that escalates gradually.

In [ ]:
wide_eval = evals_client.create(
    name="Multi-risk red-team evaluation - lab 04 optional",
    data_source_config={"type": "azure_ai_source", "scenario": "red_team"},
    testing_criteria=[
        {"type": "azure_ai_evaluator", "name": "Violence",
         "evaluator_name": "builtin.violence", "evaluator_version": "1"},
        {"type": "azure_ai_evaluator", "name": "HateUnfairness",
         "evaluator_name": "builtin.hate_unfairness", "evaluator_version": "1"},
    ],
)

wide_run = evals_client.runs.create(
    eval_id=wide_eval.id,
    name="Baseline, Base64 and ROT13",
    data_source={
        "type": "azure_ai_red_team",
        "item_generation_params": {
            "type": "red_team",
            "attack_strategies": ["baseline", "base64", "rot13"],
            "num_turns": 1,
        },
        "target": {"type": "azure_ai_model", "model": deployment_name},
    },
)

print(f"Wider run: {wide_run.id} ({wide_run.status})")

## Step 6 (optional) - The legacy preview API

`project_client.beta.red_teams` creates `/redTeams/runs` resources whose report opens the **classic**
Foundry experience. It is shown here because you may still find it in existing material, and because it
expresses the scan as a single typed `RedTeam` object.

In [ ]:
from datetime import datetime
from azure.ai.projects.models import AttackStrategy, AzureOpenAIModelConfiguration, RedTeam, RiskCategory

scan_config = RedTeam(
    target=AzureOpenAIModelConfiguration(model_deployment_name=deployment_name),
    display_name=f"lab04-legacy-{datetime.now():%Y%m%d-%H%M}",
    num_turns=1,
    attack_strategies=[AttackStrategy.BASELINE, AttackStrategy.BASE64],
    simulation_only=False,
    risk_categories=[RiskCategory.VIOLENCE],
    application_scenario="A general-purpose assistant that should refuse requests for harmful violent content.",
    tags={"purpose": "workshop-lab04"},
)

run_legacy_scan = False   # set to True to actually create the legacy scan

if run_legacy_scan:
    scan = project_client.beta.red_teams.create(red_team=scan_config)
    print(f"Scan ID: {scan.name} | status: {scan.status}")

In [ ]:
# Read the scorecard of the most recent completed legacy scan, if any
legacy_scans = [s for s in project_client.beta.red_teams.list() if (s.status or "").lower() == "completed"]

if legacy_scans:
    scan = legacy_scans[0]
    payload = scan.as_dict()
    outputs = payload.get("outputs") or {}
    raw_metrics = outputs.get("evaluationMetrics", "{}")
    metrics = json.loads(raw_metrics) if isinstance(raw_metrics, str) else raw_metrics

    print(f"Scan: {scan.display_name}")
    for label, key in (("Baseline", "violence_baseline_asr"),
                       ("Base64 (easy complexity)", "violence_easy_complexity_asr")):
        value = metrics.get(key)
        print(f"  {label:28} ASR = {'n/a' if value is None else f'{value:.0%}'}")

    report_url = (payload.get("properties") or {}).get("AiStudioEvaluationUri")
    if report_url:
        print(f"\nClassic Foundry report: {report_url}")
else:
    print("No completed legacy scan found.")

## Wrap-up

* Red teaming produces **adaptive** adversarial probes; evaluation scores a **static** dataset. You need both.
* ASR is only meaningful with its denominator and with the technique that produced it.
* Comparing baseline against transformed probes is what exposes weak controls.
* Results are evidence for mitigations and regression tests - they never prove that a system is safe.